# 05_03 Products as words: does a neural recommender beat counting?

The book's chapter ends with a trick that sounds like a joke and works: treat each invoice of an online
shop as a **sentence**, and each product in it as a **word**. Run skip-gram, and products bought together
end up close together, which is a recommender. This notebook builds it on the real data the book used,
then asks the question the book did not: is it any better than simply counting what is bought with
what?

**How this notebook works.** Every notebook in this course has the same rhythm:

1. **Recall.** Answer from memory before you look anything up. `ask()` tells you at once whether you were right.
2. **Predict, then run.** Before a cell with a surprise in it, write your prediction into `guess()`. The next cell runs the code and `reveal()` compares.
3. **Worked example, then your turn.** One example is done in full; the next, near-identical one has lines marked `# YOUR CODE HERE`.
4. **Check.** A `check_...()` cell tests what you saved, exactly as the checkpoint will, and says what to fix.

Run cells in order with **Shift+Enter**. If you get lost, **Kernel, Restart Kernel and Run All Cells** starts clean.

Running this in Google Colab? This cell sets it up; in CourseLabs it does nothing.

In [ ]:
# Colab setup. In a CourseLabs session this cell does nothing.
import os, sys
if "google.colab" in sys.modules:
    import importlib, importlib.util, subprocess
    LAB, REPO = "lab-nlp-05-words-as-points-in-space", "/content/nlp-course"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/fenago/nlp-course.git", REPO], check=True)
    os.chdir(f"{REPO}/{LAB}")
    if not os.path.exists("data"):
        os.symlink("../data", "data")
    os.makedirs("out", exist_ok=True)
    os.environ["NLPLAB_DATA"] = f"{REPO}/data"
    sys.path.insert(0, os.getcwd())
    PIP = {'torch': 'torch',
           'pandas': 'pandas',
           'numpy': 'numpy'}
    missing = [spec for mod, spec in PIP.items() if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
        importlib.invalidate_caches()
    print(f"Ready: {LAB} and its data are in {os.getcwd()}; installed {len(missing)} package(s).")
elif not os.path.isdir("/opt/nlplab/data") and os.path.isdir("data"):
    # A downloaded copy on your own computer: the helpers read data/ from here.
    os.environ["NLPLAB_DATA"] = os.path.abspath("data")

In [ ]:
import collections
import json
import os
import time
import numpy as np
from w2vtools import (load_baskets, product_vectors, cooccurrence, nearest,
                      part3_queries, part3_truth, CUTOFF, PRODUCT_SETTINGS)
from nlpcheck import ask, guess, reveal, check_05_03

start = time.time()
train_baskets, test_baskets, descriptions = load_baskets()
print(f"{len(train_baskets)} invoices before {CUTOFF}, {len(test_baskets)} after, "
      f"{len(descriptions)} products, loaded in {time.time() - start:.0f} s")
print([descriptions[c] for c in train_baskets[0][:5]])

The data is the UCI **Online Retail** set: every sale of a UK online gift shop over a year, 541,909 rows.
`load_baskets` does three things the book's notebook did not, and each is worth knowing. It drops
cancelled invoices (their numbers start with C) and returns (negative quantities), which are not
purchases. It makes each invoice a basket of distinct products. And it splits **by time**: everything
before 1 November 2011 is for learning, everything after is for testing, because a recommender's real
job is to predict purchases it has not seen yet. The book split customers at random, which lets a model
learn from the same season it is tested on.

## 1. Recall

**r5.** In 05_02, what is one training example for skip-gram? (a) a centre word and one word near it,
(b) a whole sentence, (c) a word and its definition

**r6.** Why did Gullhaven end up near Brackenridge? (a) they appear together often, (b) they share
contexts, (c) they are spelled alike

In [ ]:
ask("r5", "")
ask("r6", "")

## 2. Training on baskets

The same skip-gram as 05_02, packaged as `train_skipgram` in `w2vtools.py` so it can be reused. Products
bought at least five times before November get a vector: 48 numbers each, a window of 3, three passes
over about 1.7 million pairs. That is twelve times the pairs of 05_02, a minute or two of a session's
CPU, so it was started **in the background when your session began**, and saved to
`out/product_vectors.npz`. `product_vectors` loads that file in under a second. If you got here within
a few minutes of starting, it may not be finished; the cell then trains the vectors itself and prints
each pass, which takes two to three minutes. Either way the vectors are the same.

Precomputing like this is not a shortcut for the lab's sake: it is how real recommenders work. The
vectors are trained offline, on a schedule, and the live system only looks them up.

In [ ]:
print(PRODUCT_SETTINGS)
start = time.time()
products, pindex, P = product_vectors(train_baskets)
print(len(products), "products with vectors, ready in", round(time.time() - start), "s")

## 3. Neighbours of a product

Here is the book's example, a silver bracelet, the shop's best-seller, a hanging heart T-light holder,
and a cake stand.

In [ ]:
def similar(code, k=5):
    return [descriptions[c] for c, _ in nearest(P, products, P[pindex[code]], k, exclude=(code,))]

for code in ["90019A", "85123A", "22423"]:
    print(descriptions[code], "->", similar(code))

Jewellery with jewellery, T-light holders with T-light holders, and the Regency cake stand with its
matching teacups, teapot and sugar bowl. The model has never read a product description. It learned *heart* and
*regency* from nothing but which products customers put in the same basket.

The book goes one step further: to recommend for a **customer**, average the vectors of what they have
bought, and find the products nearest that average.

In [ ]:
basket = train_baskets[10]
customer_vector = P[[pindex[c] for c in basket if c in pindex]].mean(0)
print("bought:", [descriptions[c] for c in basket[:6]], "...")
print("suggested:", [descriptions[c] for c, _ in nearest(P, products, customer_vector, 5, exclude=tuple(basket))])

## 4. Is it any good? Measuring a recommender

"The results look relevant" is not a measurement. Here is one. Take an invoice from after November. Hide
all but one of its products, ask the recommender for ten suggestions for that one, and count a **hit** if
any of the ten was really in the invoice. The share of hits over many invoices is the **hit rate at 10**.

Two baselines make the number mean something. **Popularity** recommends the ten best-sellers to everyone.
**Co-occurrence** simply counts, for every product, how often each other product was bought in the same
invoice before November, and recommends the ten most frequent partners: no model, no training, just a
table of counts. The cell shows the counting on one invoice; `cooccurrence` in `w2vtools.py` does it for
all of them with one sparse matrix product, the same count a triple loop would make, in half a second.

`part3_queries` picks the test invoices and the product from each, with a fixed seed, so every learner
gets the same 1,000 questions here (and a different 500 in Part 3).

In [ ]:
counts = collections.Counter(c for b in train_baskets for c in b)
known = {c for c, n in counts.items() if n >= 5}
queries = part3_queries(test_baskets, known, n=1000, seed=11)
truth = part3_truth(test_baskets, known, n=1000, seed=11)
print(queries[:3], "->", [descriptions[p] for _, p in queries[:3]])

popular = [c for c, _ in counts.most_common(11)]

# Counting co-purchases, shown on one invoice: every product pairs with every other product in it.
pairs_in_one = collections.Counter((a, c) for a in train_baskets[0] for c in train_baskets[0] if a != c)
print(len(train_baskets[0]), "products in the first invoice ->", len(pairs_in_one), "co-purchase pairs")
# The same count over all 15,100 invoices, done as one sparse matrix product (see cooccurrence in w2vtools.py).
together = cooccurrence(train_baskets)

recommend = {
    "popularity": lambda p: [c for c in popular if c != p][:10],
    "cooccurrence": lambda p: together[p][:10],
    "skipgram": lambda p: [c for c, _ in nearest(P, products, P[pindex[p]], 10, exclude=(p,))],
}

**Your turn:** finish `hit_rate`. For each `(query_id, product)` in `queries`, call `recommend_fn(product)`
and count a hit if any recommendation is in `truth[query_id]`. Return the share of hits, rounded to 3
places.

In [ ]:
def hit_rate(recommend_fn, queries, truth):
    hits = 0
    # YOUR CODE HERE
    return round(hits / len(queries), 3)

Before you run the comparison, predict: will the skip-gram recommender beat plain counting of what was
bought together?

In [ ]:
guess("skipgram_beats_counting", None)   # "yes" or "no" 

In [ ]:
scores = {name: hit_rate(fn, queries, truth) for name, fn in recommend.items()}
print(scores)
reveal("skipgram_beats_counting", "yes" if scores["skipgram"] > scores["cooccurrence"] else "no")
os.makedirs("out", exist_ok=True)
json.dump(scores, open("out/05_03_eval.json", "w"), indent=1)
check_05_03()

No. Popularity scores 0.36, skip-gram 0.61, and a table of counts 0.70. This is a
common and healthy result, and worth understanding rather than hiding. With 15,000 invoices, the shop has
seen most popular pairs of products bought together many times, so the counts are already an excellent
estimate; skip-gram squeezes the same information into 48 numbers per product and loses some of the
detail. Where embeddings earn their place is where counting has nothing to say: a product bought only a
handful of times, a customer's whole history summarised as one vector (section 3), or products that are
similar without ever being bought together. Real recommenders usually combine both.

The habit to keep: **always measure against the simple baseline**. A model that cannot beat a count is
not ready, however good its neighbours look.

## 5. Exit ticket

**x2.** A hit rate at 10 of 0.70 means: (a) 70 percent of the recommendations were bought, (b) for 70
percent of the questions, at least one of the ten was bought, (c) the model is 70 percent accurate

In [ ]:
ask("x2", "")

Explain it back: name one situation where you would expect the skip-gram recommender to beat counting,
and say why.

*Your explanation:* 